In [1]:
import torch
from mmengine.config import Config, DictAction
from mmengine.runner import Runner
# from mmengine.registry import (DATA_SAMPLERS, DATASETS, EVALUATOR, FUNCTIONS,
#                                HOOKS, LOG_PROCESSORS, LOOPS, MODEL_WRAPPERS,
#                                MODELS, OPTIM_WRAPPERS, PARAM_SCHEDULERS,
#                                RUNNERS, VISUALIZERS, DefaultScope)
from mmseg.registry import MODELS


/data/JHC/openmmlab/mmengine/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \


In [2]:
# 读取权重文件
weights_aug_l = torch.load('./segnext-base-3chan.pth', map_location='cpu')
# weights_atl_mmpretrain_mae_convert = torch.load('/opt/AI-Tianlong/0-ATL-paper-work/0-预训练好的权重/1-mmpretrain-vit_large_原版_epoch_50_loss0.0009-onlybackbone.pth')
# after_convert = torch.load('/opt/AI-Tianlong/0-ATL-paper-work/0-预训练好的权重/vit-adapter/mmpretrainformat-ViT-Adapter-Aug-L_16-i21k-300ep-lr_0.001-aug_medium1-wd_0.1-do_0.1-sd_0.1--imagenet2012-steps_20k-lr_0.01-res_384.pth')

/tmp/ipykernel_380133/4025980225.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  weights_aug_l = torch.load('./segnext-base-3chan.pth', map_location='cpu')


In [3]:
with open('./seg_next_原始权重.txt', 'w') as f:
    for k in weights_aug_l['state_dict'].keys():
        f.write(k + '\t' + str(weights_aug_l['state_dict'][k].shape) + '\n') 

In [4]:
# cfg_path = '/share/home/aitlong/AI-Tianlong/OpenMMLab/mmsegmentation/configs_new/ATL-paper-test-40-对比其他经典方法/20241023-60-1-segnext_mscan-l_1xb16-adamw-160k_ade20k-512x512.py'
cfg_path = '/data/JHC/openmmlab/mmsegmentation/configs_new/segnext/JHC-crop-segnext_mscan-b-b2x4-adamw-80k_crop-512x512.py'
cfg = Config.fromfile(cfg_path)
model=cfg['model']
model_mmseg = MODELS.build(model)

/data/JHC/openmmlab/mmsegmentation/mmseg/models/backbones/mscan.py:388: UserWarning: DeprecationWarning: pretrained is deprecated, please use "init_cfg" instead
  warnings.warn('DeprecationWarning: pretrained is deprecated, '
/data/JHC/openmmlab/mmsegmentation/mmseg/models/losses/cross_entropy_loss.py:250: UserWarning: Default ``avg_non_ignore`` is False, if you would like to ignore the certain label and average loss over non-ignore labels, which is the same with PyTorch official cross_entropy, set ``avg_non_ignore=True``.
  warnings.warn(


In [5]:
model_mmseg

EncoderDecoder(
  (data_preprocessor): BaseDataPreprocessor()
  (backbone): MSCAN(
    (patch_embed1): StemConv(
      (proj): Sequential(
        (0): Conv2d(4, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (1): SyncBatchNorm(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): GELU(approximate='none')
        (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (4): SyncBatchNorm(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (block1): ModuleList(
      (0): MSCABlock(
        (norm1): SyncBatchNorm(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (attn): MSCASpatialAttention(
          (proj_1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
          (activation): GELU(approximate='none')
          (spatial_gating_unit): MSCAAttention(
            (conv0): Conv2d(64, 64, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2), groups=64)
            

In [6]:
model_mmseg.state_dict()['backbone.patch_embed1.proj.0.weight'].shape

torch.Size([32, 4, 3, 3])

In [7]:
model_mmseg.state_dict().keys()
with open('./mmseg_seg_next_b_模型中的权重.txt','w') as f:
    for key in model_mmseg.state_dict().keys():
        f.write(key + '\t' + str(model_mmseg.state_dict()[key].shape) + '\n') 

In [9]:
# 读取权重文件
weights_converted = torch.load('/data/AI-Tianlong/Checkpoints/2-对比实验的权重/segnext/base/segnext_mscan_b_10channel_BGR.pth', map_location='cpu')
# weights_atl_mmpretrain_mae_convert = torch.load('/opt/AI-Tianlong/0-ATL-paper-work/0-预训练好的权重/1-mmpretrain-vit_large_原版_epoch_50_loss0.0009-onlybackbone.pth')
# after_convert = torch.load('/opt/AI-Tianlong/0-ATL-paper-work/0-预训练好的权重/vit-adapter/mmpretrainformat-ViT-Adapter-Aug-L_16-i21k-300ep-lr_0.001-aug_medium1-wd_0.1-do_0.1-sd_0.1--imagenet2012-steps_20k-lr_0.01-res_384.pth')

In [10]:
with open('./seg_next_原始权重转换10通道后.txt', 'w') as f:
    for k in weights_converted.keys():
        f.write(k + '\t' + str(weights_converted[k].shape) + '\n') 